# Data Loaders

In [ ]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PotholeDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # normalize
        image = image.astype(np.float32) / 255.0
        print(image.shape)

        # create empty mask
        mask = np.zeros(image.shape[:-1], dtype=np.float32)

        label_path = os.path.join(self.label_dir, image_name.replace('.jpg', '.txt'))
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for l in lines:
                parts = l.strip().split()
                class_id = int(parts[0])
                coords = np.array([float(x) for x in parts[1:]]).reshape(-1, 2)

                coords[:, 0] *= image.shape[0]
                coords[:, 1] *= image.shape[0]
                coords = coords.astype(np.int32)
                cv2.fillPoly(mask, [coords], color=1.0)
        mask = np.expand_dims(mask, axis=-1)           
        return image, mask

        

In [ ]:
def getPotholeDataloader(batch_size, image_dir, label_dir):
    dataset = PotholeDataset(image_dir, label_dir)     
    return DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        drop_last=True, 
        num_workers=4
    )

In [ ]:
# Test
dataset = PotholeDataset("/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")     
loader = getPotholeDataloader(4, 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/images", 
                     "/kaggle/input/datasets/farzadnekouei/pothole-image-segmentation-dataset/Pothole_Segmentation_YOLOv8/train/labels")
dataset[0]

# Rotary Position Embeddings (RoPE)

In [ ]:
import jax
import jax.numpy as jnp 
import flax.linen as nn

def apply_rope(x, head_dim):
    # X -> B,S,H,head_dim
    seq_len = x.shape[1]
    num_heads = x.shape[2]
    dims = jnp.arange(seq_len, dtype=jnp.float32).reshape(-1, 1) # seq_len, 1
    i_2 = jnp.arange(0, head_dim, 2, dtype=jnp.float32)
    div_term = jnp.exp(-1* i_2 * jnp.log(100000)/head_dim) # head_dim//2
    freq = dims * div_term # seq_len, head_dim//2
    sin_freq = jnp.repeat(jnp.sin(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    cos_freq = jnp.repeat(jnp.cos(freq), 2, axis=-1).reshape(1, seq_len, 1, head_dim)
    odd_terms = x[..., 1::2]
    even_terms = x[..., 0::2]
    x_tmp = jnp.stack([-odd_terms, even_terms], axis=-1).reshape(x.shape)
    return x * cos_freq + x_tmp * sin_freq


In [ ]:
# test
key = jax.random.key(42)
apply_rope(jax.random.normal(key, shape=(5, 10, 12, 20)), 20)

# Multi-Head Self-Attention 

In [ ]:
class MHSA(nn.module):
    num_heads: int
    
    @nn.compact
    def __call__(self, x):
        # x -> Batch, Seq_len, dim
        b, s, dim = x.shape
        head_dim = dim//self.num_heads

        w_q = self.param('w_q', nn.initializers.lecun_normalizer(), (self.dim, self.dim))
        w_k = self.param('w_k', nn.initializers.lecun_normalizer(), (self.dim, self.dim))
        w_v = self.param('w_v', nn.initializers.lecun_normalizer(), (self.dim, self.dim))
        w_o = self.param('w_o', nn.initializers.lecun_normalizer(), (self.dim, self.dim))

        q = jnp.dot(x, w_q).reshape(b, s, self.num_heads, head_dim)
        k = jnp.dot(x, w_k).reshape(b, s, self.num_heads, head_dim)
        v = jnp.dot(x, w_v).reshape(b, s, self.num_heads, head_dim)

        qk_t = jnp.einsum("bqhd, bkhd -> bhqk", q, k) 
        atten_scores = qk_t/jnp.sqrt(head_dim)
        attention = jax.nn.softmax(atten_scores, axis=-1)
 
        att_out = jnp.einsum('bhqk, bkhd -> bqhd', attention, v).reshape(b, s, dim)
        
        out = jnp,dot(att_out, w_o)
        return out